## All the training and calculation is done on HPC, this is the archive of the code

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, AutoModelForMaskedLM
from transformers import BertForSequenceClassification
import torch
from transformers import AdamW
from transformers import get_scheduler
from transformers import Trainer
from transformers import BertTokenizer
import numpy as np
import pandas as pd
import csv
import os

#tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
tokenizer = BertTokenizer.from_pretrained("D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/chinese-bert-wwm/")

df = pd.read_csv('G:/Data/job_posting/processed/estimation/firm_sectorSample.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
batch_size = 35

In [5]:
# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
# Create a dictionary to map unique soc_codes to sequential integer labels
unique_ind_codes = sorted(df['二级行业_categoryStrMiddle'].unique())
ind_code_dict  = {二级行业_categoryStrMiddle: i for i, 二级行业_categoryStrMiddle in enumerate(unique_ind_codes)}

In [ ]:
# create into train, validation and test set

from sklearn.model_selection import train_test_split
train_df_sample, temp_df_sample = train_test_split(df, test_size=0.4, random_state=42)
valid_df_sample, test_df_sample = train_test_split(temp_df_sample, test_size=0.5, random_state=62)

# export the train, validation and test set to csv
train_df_sample.to_csv('G:/Data/job_posting/processed/finetune/train_df_Indsample.csv', index=False, encoding = "utf_8_sig", header=True)
test_df_sample.to_csv('G:/Data/job_posting/processed/finetune/test_df_Indsample.csv', index=False, encoding = "utf_8_sig", header=True)
valid_df_sample.to_csv('G:/Data/job_posting/processed/finetune/valid_df_Indsample.csv', index=False, encoding = "utf_8_sig", header=True)
 

# Generate a new column 'ind_code1' with the mapped values from '二级行业_categoryStrMiddle'
train_df_sample['ind_code1'] = train_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)
# Generate a new column 'ind_code1' with the mapped values from 'soc_code' for the test set
test_df_sample['ind_code1'] = test_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)
# Generate a new column 'ind_code1' with the mapped values from 'soc_code' for the validation set
valid_df_sample['ind_code1'] = valid_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)

C:\Users\DELL\AppData\Local\Temp/ipykernel_15152/3068623409.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df_sample['ind_code1'] = train_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)
C:\Users\DELL\AppData\Local\Temp/ipykernel_15152/3068623409.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df_sample['ind_code1'] = test_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)
C:\Users\DELL\AppData\Local\Temp/ipykernel_15152/3068623409.py:18: SettingWithCopyWarning: 
A value is t

In [13]:
# Tokenize the text and convert it into input features
train_texts = train_df_sample['经营范围_businessScope'].astype(str).tolist()
train_labels = train_df_sample['ind_code1'].tolist()

test_texts = test_df_sample['经营范围_businessScope'].astype(str).tolist()
test_labels = test_df_sample['ind_code1'].tolist()

valid_texts = valid_df_sample['经营范围_businessScope'].astype(str).tolist()
valid_labels = valid_df_sample['ind_code1'].tolist()

In [10]:
from torch.utils.data import Dataset

# Create the JobPostingDataset class
class JobPostingDataset(Dataset):
    def __init__(self, descriptions, labels, tokenizer, max_length):
        self.descriptions = descriptions
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        description = self.descriptions[idx]
        label = self.labels[idx]

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            description,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Return a tuple of the input tensors, label, and weight
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(label, dtype=torch.long),
        )

In [16]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

# Create the datasets
max_length = 512
train_dataset = JobPostingDataset(train_texts, train_labels, tokenizer, max_length)
valid_dataset = JobPostingDataset(valid_texts, valid_labels, tokenizer, max_length)
test_dataset = JobPostingDataset(test_texts, test_labels, tokenizer, max_length)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [17]:
def evaluate(model, valid_loader, device, loss_fn):
    model.eval()
    total_loss = 0
    num_batches = 0
    with torch.no_grad():
        for batch in valid_loader:
            inputs = batch[0].to(device)
            masks = batch[1].to(device)
            labels = batch[2].to(device)

            logits = model(inputs, attention_mask=masks).logits
            batch_loss = loss_fn(logits, labels)
            loss = torch.mean(batch_loss)

            total_loss += loss.item()
            num_batches += 1
    return total_loss / num_batches

In [18]:
# Define the model, the optimizer, and the learning rate scheduler
num_labels = len(train_df_sample['ind_code1'].unique())

model = BertForSequenceClassification.from_pretrained("D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/chinese-bert-wwm/", num_labels=num_labels)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
num_epochs = 30
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

import matplotlib.pyplot as plt
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights using the train_labels_np array
unique_labels = train_df_sample['ind_code1'].unique()
class_weights = compute_class_weight('balanced', classes=unique_labels, y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float) 

Some weights of the model checkpoint at D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/chinese-bert-wwm/ were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were

In [20]:
# The patience parameter determines how many consecutive epochs the model can go without an improvement in validation loss before stopping the training. 
# In this case, the patience is set to 3, meaning that if the validation loss does not improve for 3 consecutive epochs, the training will be stopped.
early_stopping_patience = 3
# This line initializes a counter variable called num_epochs_without_improvement that keeps track of the number of consecutive epochs without an improvement in validation loss. 
# The counter is set to 0 at the beginning of the training process and is incremented by 1 whenever there is no improvement in the validation loss. If the validation loss improves in a particular epoch, the counter is reset to 0.
num_epochs_without_improvement = 0
# During the training loop, if num_epochs_without_improvement becomes equal to or greater than early_stopping_patience, the training will be stopped. 
# This way, the training process can be terminated early when the model starts overfitting, or when there is no significant improvement in the validation loss.
best_valid_loss = float('inf')

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize lists to store losses
train_losses = []

# Utilize multiple GPUs with DataParallel
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

# Pass the computed class_weights to the CrossEntropyLoss function:
# Create a loss function that doesn't reduce the losses right away and pass class_weights
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device), reduction='none')
# By incorporating class weights into the loss function, the model will pay more attention to the minority classes during training. 

model.train()
for epoch in range(num_epochs):
    epoch_train_loss = 0
    num_batches = 0
    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()

        logits = model(inputs, attention_mask=masks).logits

        # Compute the loss for each sample
        batch_loss = loss_fn(logits, labels)

        # Average the weighted losses
        loss = torch.mean(batch_loss)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # Add the current batch loss to the epoch_train_loss
        epoch_train_loss += loss.item()
        num_batches += 1

    # Calculate average loss for the current epoch and append it to the train_losses list
    epoch_train_loss /= num_batches
    train_losses.append(epoch_train_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_train_loss:.4f}")

    # Evaluate the model on the validation set
    valid_loss = evaluate(model, valid_loader, device, loss_fn)
    print(f"Validation Loss: {valid_loss:.4f}")

    # Save the best model based on the validation loss
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        if isinstance(model, torch.nn.DataParallel):
            model.module.save_pretrained("G:/Data/job_posting/processed/finetune/best_model_Indsample")
        else:
            model.save_pretrained("G:/Data/job_posting/processed/finetune/best_model_Indsample")
        num_epochs_without_improvement = 0
    else:
        num_epochs_without_improvement += 1

    # Check the stopping condition and break the loop if needed
    if num_epochs_without_improvement >= early_stopping_patience:
        print("Early stopping due to no improvement in validation loss.")
        break

In [ ]:
actual_num_epochs = len(train_losses)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, actual_num_epochs + 1), train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
# Save the figure
plt.savefig('/share/home/320346/trained_model/Indsample_training_loss_plot.pdf', bbox_inches='tight')
plt.close()


# Evaluate the performance of the model on the test set
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)

        # Resize the attention mask tensor to match the size of the inputs
        # masks.resize_(inputs.shape[0], inputs.shape[1])
    
        outputs = model(inputs, attention_mask=masks)
        logits = outputs.logits
        batch_predictions = torch.argmax(logits, axis=1).cpu().numpy()
        predictions.extend(batch_predictions)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_labels, predictions)
print(f"Accuracy: {accuracy}")

In [ ]:
# First, we need to modify the true_labels and predictions by extracting the major SOC groups from the original SOC codes.
# Create a list of SOC codes ordered by their corresponding sequential labels
ordered_ind_codes = [二级行业_categoryStrMiddle for 二级行业_categoryStrMiddle, _ in sorted(ind_code_dict.items(), key=lambda item: item[1])]

# Convert test_labels and predictions back to SOC codes
test_labels_ind = np.array([ordered_ind_codes[label] for label in test_labels])
predictions_ind = np.array([ordered_ind_codes[label] for label in predictions])

# Calculate accuracy for each major SOC group
major_group_accuracies = {}
unique_major_groups = np.unique(test_labels_ind)
for major_group in unique_major_groups:
    major_group_indices = np.where(test_labels_ind == major_group)
    major_group_accuracy = accuracy_score(test_labels_ind[major_group_indices], predictions_ind[major_group_indices])
    major_group_accuracies[major_group] = major_group_accuracy
    print(f"Major Group {major_group}: Accuracy: {major_group_accuracy:.4f}")

major_group_accuracies_df = pd.DataFrame(list(major_group_accuracies.items()), columns=['Major Group', 'Accuracy'])
major_group_accuracies_df.to_csv('/share/home/320346/trained_model/Indmajor_group_accuracies.csv', index=False)

### After the training is done, we call the predict function to predict the class of the data

In [ ]:
import os
import pandas as pd
import torch
import gc
from transformers import BertTokenizer, BertForSequenceClassification

# Load the fine-tuned model and tokenizer
tokenizer = BertTokenizer.from_pretrained('/share/home/320346/bert-base-chinese/')
model = BertForSequenceClassification.from_pretrained('/share/home/320346/best_model_Indsample/')

# Set the device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# Check if there are multiple GPUs available
if torch.cuda.device_count() > 1:
    print(f"Let's use {torch.cuda.device_count()} GPUs!")
    # If so, wrap the current model in nn.DataParallel to use multiple GPUs.
    model = torch.nn.DataParallel(model)

# Load the mappings
df = pd.read_csv("/share/home/320346/firm_sectorSample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

unique_ind_codes = sorted(df['二级行业_categoryStrMiddle'].unique())
ind_code_dict  = {二级行业_categoryStrMiddle: i for i, 二级行业_categoryStrMiddle in enumerate(unique_ind_codes)}
inverse_ind_code_dict = {v: k for k, v in ind_code_dict.items()}

del df
gc.collect()

batch_size = 1200  # Adjust as necessary

df = pd.read_csv('/share/home/320346/mapped_ind_sector/sectorMissing.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
num_examples = len(df['经营范围_businessScope'])

for i in range(0, num_examples, batch_size):
    batch_text = df['经营范围_businessScope'][i:i+batch_size].astype(str).tolist()
    inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=512, return_tensors='pt')

    inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)
    predictions_list = predictions.cpu().numpy().tolist()
    original_ind_codes = [inverse_ind_code_dict[pred] for pred in predictions_list]

    df.loc[i:i+batch_size-1, 'predicted_ind_code'] = original_ind_codes

    torch.cuda.empty_cache()
    
df.to_csv('/share/home/320346/predict_indSector.csv', index=False, encoding='utf_8_sig')